# Resultados: world models de conducción sobre tokens de commaVQ
Este notebook muestra los resultados y muestras del proyecto **sin reentrenar nada**: lee `results/` (métricas, figuras, rollouts guardados) y solo carga un modelo si querés generar en vivo (sección 7). El informe completo está en `Docs/resultados.md`.

Corre en local (Jupyter o VS Code, con cualquier directorio de trabajo dentro del repo) y en Colab (clona el repo si no está).

## 1. Preparación

In [ ]:
%matplotlib inline
import json, os, subprocess, sys
from pathlib import Path

def raiz_repo(p=None):
  """Sube desde p (por defecto el cwd) hasta encontrar src/common.py."""
  p = Path(p or os.getcwd()).resolve()
  for d in [p, *p.parents]:
    if (d / "src" / "common.py").exists():
      return d
  return None

def correr(args, cwd=None):
  """Corre un comando mostrando la salida a medida que sale (igual en Windows, Linux y Colab)."""
  p = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                       encoding="utf-8", errors="replace")
  for linea in p.stdout:
    print(linea, end="")
  if p.wait():
    raise RuntimeError(f"falló: {' '.join(map(str, args))}")

REPO = raiz_repo()
if REPO is None:  # Colab (o cualquier lugar sin el repo): se clona e instala como en notebooks/colab.ipynb
  os.chdir("/content" if Path("/content").exists() else ".")
  correr(["git", "clone", "https://github.com/bpeco/ml2-gen-world-model.git"])
  os.chdir("ml2-gen-world-model")
  correr(["git", "clone", "--depth", "1", "https://github.com/commaai/commavq.git", "third_party/commavq"])
  correr([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements-metrics.txt"])
  os.environ["HF_HUB_DISABLE_XET"] = "1"  # las descargas con xet se colgaban
  correr([sys.executable, "-m", "src.data"])  # val, pesos del gpt2m y decoder
  REPO = raiz_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
  sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, Markdown, display
from src import common, ejemplos
print("repo:", REPO)
print(common.device_info())

> Los rollouts de `results/rollouts/` y los checkpoints no están en git. Si faltan (por ejemplo en Colab), se bajan del almacén compartido con `python -m src.store pull --rollouts` (hay que indicar el backend con `--dir RUTA` o `--hf REPO`, ver `src/store.py`).

## 2. Tabla resumen
Una fila por configuración, con las métricas del informe: cross-entropy en val (`CE`, y la del último frame), LPIPS y FVD a 1 s (con IC 95 %), FVD a 3 s y latencia por frame (batch 1, fp16, RTX 3080 Ti). La estrella marca la frontera de Pareto calidad vs. velocidad.

In [ ]:
import html

summary = json.loads(Path("results/summary.json").read_text(encoding="utf-8"))["configs"]

def n(x, d=3):
  return "–" if x is None else f"{x:.{d}f}".replace(".", ",")

def fila(c):
  ci = c.get("fvd_1s_ci")
  fvd = "–" if c["fvd_1s"] is None else f"{c['fvd_1s']:.0f}" + (f" [{ci[0]:.0f}, {ci[1]:.0f}]" if ci else "")
  rt = {True: "sí", False: "no", None: "–"}[c["realtime"]]
  ms = c["ms_frame"]
  ms = "–" if ms is None else (f"{ms:.0f}" if ms >= 10 else n(ms, 1 if ms >= 1 else 2))
  params = "–" if c["params_M"] is None else f"{c['params_M']:.1f}M".replace(".", ",")
  return [c["label"] + (" ★" if c["pareto"] else ""), c["family"], params, n(c["ce_all"]), n(c["ce_last"]),
          n(c["lpips_1s"]), fvd, n(c["fvd_3s"], 0), ms, rt]

COLS = ["Configuración", "Familia", "Params", "CE", "CE último frame", "LPIPS 1 s", "FVD 1 s [IC 95 %]", "FVD 3 s",
        "ms/frame", "< 50 ms"]

def tabla_html(cols, filas, texto_hasta=2):
  """Tabla HTML; las primeras `texto_hasta` columnas a la izquierda, el resto a la derecha."""
  def celda(tag, i, v, borde):
    al = "left" if i < texto_hasta else "right"
    return f"<{tag} style='text-align:{al};padding:3px 10px;border-bottom:{borde}'>{html.escape(str(v))}</{tag}>"
  h = "<table style='border-collapse:collapse;font-size:13px'><thead><tr>"
  h += "".join(celda("th", i, c, "2px solid #888") for i, c in enumerate(cols)) + "</tr></thead><tbody>"
  for f in filas:
    h += "<tr>" + "".join(celda("td", i, v, "1px solid #ddd") for i, v in enumerate(f)) + "</tr>"
  return HTML(h + "</tbody></table>")

for titulo, grupos in [("Configuraciones principales", ("principal",)), ("Extras y ablaciones", ("extra", "ablación"))]:
  display(Markdown(f"**{titulo}**"))
  display(tabla_html(COLS, [fila(c) for c in summary if c["group"] in grupos]))

### Hipótesis
Resumen de las pruebas estadísticas (`results/hypotheses.md`).

In [ ]:
display(Markdown(Path("results/hypotheses.md").read_text(encoding="utf-8")))

## 3. Figuras
Las figuras salen de `results/` (las genera `python -m src.report`; acá solo se muestran).

In [ ]:
FIGURAS = [
  ("fig_pareto.png", "**Calidad vs. latencia.** FVD a 1 s contra ms por frame (batch 1, fp16 + CUDA graphs). La frontera de Pareto se calcula entre las configuraciones con 100M tokens; ConvLSTM con 1B tokens se muestra aparte."),
  ("fig_pareto_lpips.png", "**Lo mismo con LPIPS a 1 s** en el eje vertical."),
  ("fig_horizon.png", "**Calidad según el horizonte.** El frame congelado tiene mejor LPIPS que todos los modelos chicos y el peor FVD entre ellos: LPIPS pareado premia no moverse, por eso el realismo se juzga con FVD."),
  ("fig_ce_vs_fvd.png", "**CE vs. FVD.** Entre las configuraciones autoregresivas la CE (barata) ordena igual que el FVD a 1 s (Spearman 1,00)."),
  ("fig_latency.png", "**Latencia por optimización.** fp16 solo apenas acelera a batch 1; con CUDA graphs la aceleración es de 2,8 a 3,1x en los transformers."),
  ("fig_ce_posicion.png", "**CE por posición en la ventana de 20 frames.** Con embedding de posición absoluta el GPT-S empeora en los últimos frames (justo donde genera); con frame + celda la curva queda plana."),
]
for f, texto in FIGURAS:
  if Path("results", f).exists():
    display(Markdown(texto))
    display(Image(filename=str(Path("results", f))))
  else:
    print("falta results/" + f)

## 4. Modelos disponibles
Corridas registradas en `results/checkpoints.json`, si su `checkpoints/<run>/final.pt` está en esta máquina, y los rollouts guardados en `results/rollouts/`.

In [ ]:
manifiesto = json.loads(Path("results/checkpoints.json").read_text(encoding="utf-8"))
runs = {k: v for k, v in manifiesto.items() if not k.startswith("_")}  # "_rollouts" es otra sección

def cfg(v):
  c = v.get("config", {})
  return ", ".join(f"{k}={c[k]}" for k in ("n_layer", "dim", "pos", "hidden", "n_layers") if k in c)

filas = [[r, v["kind"], f"{v['params'] / 1e6:.1f}M".replace(".", ","), f"{v['tokens'] / 1e6:.0f}M", cfg(v),
          "sí" if Path("checkpoints", r, "final.pt").exists() else "no"] for r, v in sorted(runs.items())]
display(tabla_html(["Corrida", "Tipo", "Params", "Tokens vistos", "Config", "final.pt local"], filas))
roll = ejemplos.disponibles()
print(f"\n{len(roll)} rollouts guardados (400 clips x 60 frames cada uno):")
print(", ".join(roll))

## 5. Ejemplos de los rollouts guardados
Cada modelo recibe 1 s de contexto real y genera 3 s. Los clips por defecto se eligen solo por cuánto se mueve la escena real, no por cómo le va a un modelo.

Cambiá `CLIP` y `MODELOS` para ver otra cosa. Los percentiles 50 / 75 / 90 de movimiento son los clips 209 / 258 / 179 (`elegir_clips()[0]`, `[1]`, `[2]`), y se puede poner cualquier clip de 0 a 399. Los nombres válidos de `MODELOS` son los de la lista de la sección 4. Desde la terminal, `python -m src.ejemplos` genera los PNG y GIF de los tres clips en `results/ejemplos/`.

In [ ]:
CLIP = ejemplos.elegir_clips()[0]  # percentiles 50/75/90 = clips 209/258/179; vale cualquiera de 0 a 399
MODELOS = ["real", "gpt2m", "gpt_s", "maskgit_s_s4", "convlstm", "convlstm_1b", "copy"]

ctx_img, imgs = ejemplos.clip(CLIP, MODELOS)
titulo, _ = ejemplos.titulo_clip(CLIP)
fig = ejemplos.grilla(ctx_img, imgs, titulo)
plt.show()

In [ ]:
# GIF con todos los modelos lado a lado: contexto real y después lo generado, a velocidad real
display(Image(data=ejemplos.gif(ctx_img, imgs), format="gif"))

## 6. Curvas LPIPS por horizonte
`results/cache/lpips_<rollout>.npy` guarda el LPIPS contra el video real de cada clip y cada frame generado (400 x 60). A la izquierda, la media de los 400 clips; a la derecha, el clip elegido arriba. Menos es mejor (pero ojo: LPIPS premia la cautela, ver el frame congelado).

In [ ]:
COLORES = {"gpt": "#2a78d6", "maskgit": "#eb6834", "convlstm": "#1baf7a", "gpt2m": "#3d3d3d", "base": "#9a9a9a"}
ESTILOS = ["-", "--", ":", "-."]

def familia(nombre):
  if nombre.startswith("gpt2m"):
    return "gpt2m"
  for f in ("gpt", "maskgit", "convlstm"):
    if nombre.startswith(f):
      return f
  return "base"  # real, copy, frozen, unigram

curvas = {}
for m in [m for m in MODELOS if m != "real"]:  # real vale 0 por definición
  p = Path("results/cache") / f"lpips_{m}.npy"
  if p.exists():
    curvas[m] = np.load(p)  # (400, 60)
  else:
    print(f"sin curva LPIPS para {m} ({p})")

# mismo color por familia; si hay dos de la misma familia se distinguen con el estilo de línea
cuenta, est = {}, {}
for m in curvas:
  f = familia(m)
  cuenta[f] = cuenta.get(f, -1) + 1
  est[m] = (COLORES[f], ESTILOS[cuenta[f] % len(ESTILOS)])

t = (np.arange(60) + 1) / common.FPS
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
paneles = [("Media de los 400 clips", lambda a: a.mean(0)), (f"Clip {CLIP}", lambda a: a[CLIP])]
for ax, (titulo, sel) in zip(axs, paneles):
  for m, a in curvas.items():
    c, ls = est[m]
    ax.plot(t, sel(a), color=c, ls=ls, lw=1.8, label=ejemplos.etiqueta(m)[0])
  ax.set_title(titulo, fontsize=11)
  ax.set_xlabel("segundos generados")
  ax.grid(alpha=0.25)
  for s in ("top", "right"):
    ax.spines[s].set_visible(False)
axs[0].set_ylabel("LPIPS vs. real")
axs[1].legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()

## 7. Generar en vivo con un modelo entrenado
Carga un checkpoint, genera 3 s a partir del contexto real de `CLIP` y lo compara con el video real y con el rollout guardado del mismo modelo (que se hizo por lotes de 50 clips con CUDA graphs y otra semilla, así que no es idéntico: es otra muestra del mismo modelo). Solo hace falta un modelo chico en la GPU (~2 GB).

Parámetros: `RUN` es una corrida de la sección 4; `STEPS` solo cuenta para MaskGIT (pasos de desenmascarado); `DTYPE` es `fp32`, `fp16` o `bf16`.

In [ ]:
RUN = "maskgit_s"
STEPS = 4
TEMPERATURE = 1.0
SEED = 0
DTYPE = "fp16"
HORIZON = 60

In [ ]:
import time
import torch
from src.models import load_model
from src.rollouts import DTYPES

generado = None
kind = runs[RUN]["kind"] if RUN in runs else None
if not Path("checkpoints", RUN, "final.pt").exists():
  print(f"No está checkpoints/{RUN}/final.pt en esta máquina. Se baja del almacén compartido con:\n"
        f"  python -m src.store pull {RUN} --dir RUTA     (o --hf REPO; ver src/store.py)\n"
        f"y después se vuelve a correr esta celda. Mientras tanto, la sección 5 muestra los rollouts guardados.")
else:
  try:
    dev = common.device()
    steps = STEPS if kind == "maskgit" else None  # steps solo lo usan los MaskGIT
    model = load_model(RUN, dev).eval().to(DTYPES[DTYPE])
    ctx, _ = ejemplos.contexto(CLIP)
    ctx = torch.as_tensor(np.asarray(ctx).astype(np.int64), device=dev)[None]  # (1, 20, 128)

    def sync():
      if dev.type == "cuda":
        torch.cuda.synchronize()

    kw = dict(temperature=TEMPERATURE, steps=steps)
    model.rollout(ctx, 2, generator=torch.Generator(device=dev).manual_seed(SEED), **kw)  # calentamiento
    g = torch.Generator(device=dev).manual_seed(SEED)
    sync()
    t0 = time.time()
    r = model.rollout(ctx, HORIZON, generator=g, **kw)
    sync()
    seg = time.time() - t0
    generado = r[0].cpu().numpy().astype(np.int16)
    modo = f"steps={steps}" if steps else "autoregresivo"
    print(f"{RUN} ({DTYPE}, {modo}): {HORIZON} frames en {seg:.2f} s = {1000 * seg / HORIZON:.1f} ms/frame "
          f"(batch 1, sin decoder ni CUDA graphs) en {common.device_info(dev).get('gpu', dev.type)}")
  except Exception as e:
    print(f"No se pudo generar con {RUN}: {type(e).__name__}: {e}")

In [ ]:
if generado is not None:
  guardado = f"{RUN}_s{STEPS}" if kind == "maskgit" else RUN
  modelos = ["real"] + ([guardado] if (ejemplos.ROLL / f"{guardado}.npy").exists() else [])
  ctx_img_v, imgs_v = ejemplos.clip(CLIP, modelos, generados={f"{RUN} (en vivo)": generado})
  ejemplos.grilla(ctx_img_v, imgs_v, f"Clip {CLIP}: real, rollout guardado y generado en vivo")
  plt.show()
  display(Image(data=ejemplos.gif(ctx_img_v, imgs_v, cols=3), format="gif"))